# 05 — Pro Consumidor Prices | Bronze Ingestion

Ingest the Pro Consumidor "Precios Productos de Primera Necesidad, por segmento de comercialización" file as published.

| Item | Value |
|---|---|
| File | `precios de productos de primera necesidad.xlsx` (the CSV has inconsistent delimiters and is not used) |
| Layout | Title on row 1 (states the report period), headers on row 3: Id, Producto, Tipo, Precio, Mes, Año, establecimiento |
| Target | `rd_economic_intelligence.bronze.proconsumidor_prices_raw` |

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone
import uuid, json
import pandas as pd
from pyspark.sql import functions as F

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/proconsumidor_raw/consumer_prices/"
FILE = "precios de productos de primera necesidad.xlsx"
TARGET = f"{CATALOG}.bronze.proconsumidor_prices_raw"
EXPECTED = ["Id", "Producto", "Tipo", "Precio", "Mes", "Año", "establecimiento"]
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

info = {f.name: f for f in dbutils.fs.ls(RAW_PATH)}.get(FILE)
if info is None:
    raise FileNotFoundError(RAW_PATH + FILE)

raw = pd.read_excel(RAW_PATH + FILE, header=None, dtype=object)
title = str(raw.iloc[0, 0]).strip()
header_row = next(i for i in range(10) if [str(v).strip() for v in raw.iloc[i, :7]] == EXPECTED)
data = raw.iloc[header_row + 1:, :7]
data.columns = EXPECTED
data = data.dropna(how="all")
print(f"Report title: {title}\nHeader row: {header_row + 1} · data rows: {len(data)}")

rows = [(i + 1, *[None if pd.isna(v) else str(v) for v in r]) for i, r in zip(data.index, data.itertuples(index=False))]
df = (spark.createDataFrame(rows, "_source_row_number int, id string, producto string, tipo string, precio string, mes string, anio string, establecimiento string")
      .withColumn("_report_title", F.lit(title)).withColumn("_source_file", F.lit(FILE))
      .withColumn("_source_modified_ts", F.lit(datetime.fromtimestamp(info.modificationTime / 1000, tz=timezone.utc)).cast("timestamp"))
      .withColumn("_ingestion_ts", F.lit(INGESTION_TS).cast("timestamp")).withColumn("_batch_id", F.lit(BATCH_ID)))

if spark.catalog.tableExists(TARGET):
    spark.sql(f"DELETE FROM {TARGET} WHERE _source_file = '{FILE}'")
df.write.format("delta").mode("append").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: Pro Consumidor basic-goods prices by retail segment, as published (text values).'")
n = spark.table(TARGET).filter(F.col("_batch_id") == BATCH_ID).count()
assert n == len(rows)
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "rows": n, "title": title}, ensure_ascii=False))